# 🚀 AI Subtitle Studio — Google Colab Real AI Acceleration Backend

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/margacis2/ai-subtitle-studio/blob/main/colab_backend.ipynb)

**No paid AI API keys required.** This notebook runs real **Faster-Whisper** and **OPUS-MT** in your Google Colab runtime and exposes it securely to your local **AI Subtitle Studio** web application.

---

## 📖 Run in this order:
1. In Google Colab, select **Runtime** → **Change runtime type** → **T4 GPU** (if available) → **Save**.
2. Run **Cell 1** (Installs pinned dependencies, FFmpeg, and espeak-ng).
3. Run **Cell 2** (Configures the Faster-Whisper + OPUS-MT backend engine).
4. Run **Cell 3** (Starts the API server & public HTTPS tunnel).
5. Copy the **Engine URL** and **API Token** into your local **AI Subtitle Studio** (Settings → Colab Addon).
6. *(Optional)* Run **Cell 4** to execute an end-to-end real speech self-test inside Colab.

> **Note:** Colab sessions are temporary. When your runtime disconnects or is stopped, the temporary URL goes offline.

In [ ]:
# @title 1. Install dependencies (run once per fresh Colab runtime)
import os, sys, subprocess, shutil

def run(cmd):
    print("$", " ".join(cmd))
    subprocess.check_call(cmd)

# Ensure system audio tools are present
if not shutil.which("ffmpeg") or not shutil.which("espeak-ng"):
    print("[*] Installing system audio packages (ffmpeg, espeak-ng)...")
    run(["apt-get", "update", "-qq"])
    run(["apt-get", "install", "-y", "-qq", "ffmpeg", "espeak-ng"])

# Install pinned, compatible Python dependencies without upgrading entire PyTorch environment
print("[*] Installing Faster-Whisper, HuggingFace Transformers, FastAPI & utilities...")
run([sys.executable, "-m", "pip", "install", "-q",
     "faster-whisper==1.2.1",
     "fastapi>=0.110,<1", "uvicorn[standard]>=0.27,<1",
     "python-multipart>=0.0.9", "pyngrok>=7,<8",
     "transformers>=4.40,<5", "sentencepiece", "sacremoses",
     "requests"])

# Install Cloudflare Tunnel binary for instant 0-config HTTPS tunnels without login/password challenges
if not shutil.which("cloudflared"):
    print("[*] Installing Cloudflare Tunnel (cloudflared)...")
    subprocess.run("wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb && dpkg -i cloudflared-linux-amd64.deb > /dev/null 2>&1 && rm -f cloudflared-linux-amd64.deb", shell=True)

import torch, faster_whisper, fastapi, uvicorn
print("\n" + "=" * 55)
print("✅ ALL DEPENDENCIES INSTALLED SUCCESSFULLY")
print("=" * 55)
print("Python        :", sys.version.split()[0])
print("PyTorch       :", torch.__version__)
print("Faster-Whisper:", getattr(faster_whisper, "__version__", "installed"))
if torch.cuda.is_available():
    vram = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    print(f"GPU           : [ACTIVE] {torch.cuda.get_device_name(0)} ({vram:.1f} GB VRAM)")
else:
    print("GPU           : [INACTIVE] CPU fallback mode")
print("FFmpeg        :", shutil.which("ffmpeg"))
print("Cloudflared   :", shutil.which("cloudflared"))
print("=" * 55)

In [ ]:
# @title 2. Configure and create the real AI backend
import os, time, secrets, tempfile, threading, gc
from pathlib import Path
from typing import Optional, List
import torch
from fastapi import FastAPI, UploadFile, File, Form, Header, HTTPException, Depends
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel, Field
from faster_whisper import WhisperModel

# ---- SETTINGS ----
WHISPER_MODEL = "small"  # @param ["tiny", "base", "small", "medium", "large-v3", "large-v3-turbo"]
MAX_UPLOAD_MB = 95       # @param {type:"integer"}
API_TOKEN = secrets.token_urlsafe(24)  # Automatically generated secure token per runtime

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
COMPUTE_TYPE = "float16" if DEVICE == "cuda" else "int8"

print("AI Model      :", WHISPER_MODEL)
print("Default Device:", DEVICE)
print("Upload Limit  :", MAX_UPLOAD_MB, "MB")
print("API Token     :", API_TOKEN)

_whisper = None
_whisper_lock = threading.Lock()
_translation_cache = {}
_translation_lock = threading.Lock()

def get_whisper():
    global _whisper, DEVICE, COMPUTE_TYPE
    with _whisper_lock:
        if _whisper is not None:
            return _whisper
        print(f"[*] Loading Faster-Whisper '{WHISPER_MODEL}'...")
        started = time.time()
        # Multi-stage fallback: float16 -> int8_float16 -> int8 on CUDA, then CPU
        if torch.cuda.is_available():
            for compute in ("float16", "int8_float16", "int8"):
                try:
                    _whisper = WhisperModel(WHISPER_MODEL, device="cuda", compute_type=compute)
                    DEVICE, COMPUTE_TYPE = "cuda", compute
                    break
                except Exception as exc:
                    print(f"CUDA {compute} unavailable: {str(exc)[:160]}")
            if _whisper is None:
                print("CUDA model initialization failed; falling back to CPU.")
        if _whisper is None:
            _whisper = WhisperModel(
                WHISPER_MODEL,
                device="cpu",
                compute_type="int8",
                cpu_threads=max(2, (os.cpu_count() or 4) // 2)
            )
            DEVICE, COMPUTE_TYPE = "cpu", "int8"
        print(f"✅ Faster-Whisper loaded in {time.time()-started:.1f}s ({DEVICE}/{COMPUTE_TYPE})")
        return _whisper

def check_token(authorization: Optional[str] = Header(default=None)):
    if not API_TOKEN:
        return True
    if not authorization or not authorization.lower().startswith("bearer "):
        raise HTTPException(status_code=401, detail="Send Authorization: Bearer <API_TOKEN>")
    supplied = authorization.split(" ", 1)[1].strip()
    if not secrets.compare_digest(supplied, API_TOKEN):
        raise HTTPException(status_code=403, detail="Invalid API token")
    return True

app = FastAPI(title="AI Subtitle Studio Colab Engine", version="3.0.0")
app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_credentials=False,
    allow_methods=["GET", "POST", "OPTIONS"],
    allow_headers=["Authorization", "Content-Type"],
)

@app.get("/")
def root():
    return {"name": "AI Subtitle Studio Colab Engine", "status": "online", "docs": "/docs"}

@app.get("/health")
def health():
    gpu = torch.cuda.get_device_name(0) if torch.cuda.is_available() else None
    return {
        "status": "online",
        "whisper_model": WHISPER_MODEL,
        "device": DEVICE,
        "compute_type": COMPUTE_TYPE,
        "gpu": gpu,
        "max_upload_mb": MAX_UPLOAD_MB,
        "has_token": True
    }

@app.get("/system")
def system():
    return health()

@app.get("/models")
def models():
    return {
        "models": ["tiny", "base", "small", "medium", "large-v3", "large-v3-turbo"],
        "selected": WHISPER_MODEL,
        "device": DEVICE
    }

class TranslateBody(BaseModel):
    texts: List[str] = Field(..., max_length=128)
    source_lang: str = "ja"
    target_lang: str = "en"

# Explicit validated OPUS-MT translation pairs (no simulated fallback)
OPUS_MODELS = {
    ("ja", "en"): "Helsinki-NLP/opus-mt-ja-en",
    ("fr", "en"): "Helsinki-NLP/opus-mt-fr-en",
    ("de", "en"): "Helsinki-NLP/opus-mt-de-en",
    ("es", "en"): "Helsinki-NLP/opus-mt-es-en",
    ("zh", "en"): "Helsinki-NLP/opus-mt-zh-en",
    ("ar", "en"): "Helsinki-NLP/opus-mt-ar-en",
    ("hi", "en"): "Helsinki-NLP/opus-mt-hi-en",
    ("ta", "en"): "Helsinki-NLP/opus-mt-dra-en",
    ("te", "en"): "Helsinki-NLP/opus-mt-dra-en",
    ("ml", "en"): "Helsinki-NLP/opus-mt-dra-en",
    ("kn", "en"): "Helsinki-NLP/opus-mt-dra-en",
}

def translate_opus(texts, source_lang, target_lang):
    src = source_lang.lower().split("-")[0]
    tgt = target_lang.lower().split("-")[0]
    if src == tgt:
        return texts
    pair = (src, tgt)
    model_id = OPUS_MODELS.get(pair)
    if not model_id:
        raise HTTPException(
            status_code=400,
            detail=f"Translation pair {src} → {tgt} is not configured. Supported: {', '.join(a+'→'+b for a,b in OPUS_MODELS)}"
        )
    with _translation_lock:
        if model_id not in _translation_cache:
            from transformers import MarianMTModel, MarianTokenizer
            print("[*] Loading neural translation model:", model_id)
            tokenizer = MarianTokenizer.from_pretrained(model_id)
            model = MarianMTModel.from_pretrained(model_id)
            if DEVICE == "cuda":
                model = model.to("cuda")
            model.eval()
            _translation_cache[model_id] = (tokenizer, model)
        tokenizer, model = _translation_cache[model_id]

    output = []
    for start in range(0, len(texts), 8):
        original_batch = texts[start : start + 8]
        batch = [x if x.strip() else "." for x in original_batch]
        inputs = tokenizer(batch, return_tensors="pt", padding=True, truncation=True, max_length=512)
        if DEVICE == "cuda":
            inputs = {k: v.to("cuda") for k, v in inputs.items()}
        with torch.inference_mode():
            generated = model.generate(**inputs, max_new_tokens=256)
        decoded = tokenizer.batch_decode(generated, skip_special_tokens=True)
        output.extend("" if not orig.strip() else trans for orig, trans in zip(original_batch, decoded))
    return output

@app.post("/translate")
def translate_api(body: TranslateBody, authorized: bool = Depends(check_token)):
    translations = translate_opus(body.texts, body.source_lang, body.target_lang)
    return {"source_lang": body.source_lang, "target_lang": body.target_lang, "translations": translations}

@app.post("/transcribe")
async def transcribe_api(
    file: UploadFile = File(...),
    source_lang: str = Form("auto"),
    target_lang: str = Form("original"),
    word_timestamps: bool = Form(False),
    authorized: bool = Depends(check_token)
):
    suffix = Path(file.filename or "audio.wav").suffix[:10] or ".audio"
    tmp_path = None
    size = 0
    try:
        with tempfile.NamedTemporaryFile(delete=False, suffix=suffix) as tmp:
            tmp_path = tmp.name
            while True:
                chunk = await file.read(1024 * 1024)
                if not chunk:
                    break
                size += len(chunk)
                if size > MAX_UPLOAD_MB * 1024 * 1024:
                    raise HTTPException(
                        status_code=413,
                        detail=f"File exceeds {MAX_UPLOAD_MB} MB limit ({size/(1024*1024):.1f} MB received). Convert or split audio below 95 MB before uploading."
                    )
                tmp.write(chunk)
        if size == 0:
            raise HTTPException(status_code=400, detail="Uploaded file is empty.")

        lang = None if source_lang.strip().lower() in ("", "auto") else source_lang.strip().split("-")[0]
        model = get_whisper()
        started = time.time()
        segments_gen, info = model.transcribe(
            tmp_path,
            language=lang,
            task="transcribe",
            beam_size=5,
            vad_filter=True,
            word_timestamps=word_timestamps,
            condition_on_previous_text=False
        )
        segments = list(segments_gen)
        original = [seg.text.strip() for seg in segments]
        translated = original
        if target_lang.lower() not in ("original", "same", info.language.lower()):
            translated = translate_opus(original, info.language, target_lang)

        result_segments = []
        for seg, orig, trans in zip(segments, original, translated):
            if not orig:
                continue
            item = {
                "startMs": round(seg.start * 1000),
                "endMs": round(seg.end * 1000),
                "original": orig,
                "translation": trans,
                "text": orig if target_lang.lower() == "original" else trans,
                "confidence": round((1.0 - float(getattr(seg, "no_speech_prob", 0.0))) * 100),
                "lang": info.language
            }
            if word_timestamps and getattr(seg, "words", None):
                item["words"] = [{"startMs": round(w.start * 1000), "endMs": round(w.end * 1000), "text": w.word} for w in seg.words]
            result_segments.append(item)

        elapsed = time.time() - started
        return {
            "language": info.language,
            "segments": result_segments,
            "duration": info.duration,
            "processing_seconds": round(elapsed, 2),
            "device": DEVICE,
            "model": WHISPER_MODEL
        }
    except HTTPException:
        raise
    except Exception as exc:
        raise HTTPException(status_code=500, detail=f"Transcription failed: {type(exc).__name__}: {exc}")
    finally:
        await file.close()
        if tmp_path and os.path.exists(tmp_path):
            try: os.remove(tmp_path)
            except OSError: pass
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

print("✅ Backend configured successfully. Ready to start tunnel.")

In [ ]:
# @title 3. Start API Server & Public Tunnel
import time, socket, threading, requests, subprocess, re
import uvicorn

# @markdown ### Choose Tunnel Provider
# @markdown **Cloudflare (Recommended)**: Direct HTTPS, no account needed, no password prompt!
# @markdown **Ngrok**: Enter your free Ngrok auth token if using Ngrok.
TUNNEL_PROVIDER = "Cloudflare (0-Config, Recommended)" # @param ["Cloudflare (0-Config, Recommended)", "Ngrok"]
NGROK_AUTH_TOKEN = "" # @param {type:"string"}

def port_is_open(port):
    with socket.socket() as sock:
        return sock.connect_ex(("127.0.0.1", port)) == 0

PORT = 8001
if not port_is_open(PORT):
    config = uvicorn.Config(app, host="0.0.0.0", port=PORT, log_level="warning")
    server = uvicorn.Server(config)
    server_thread = threading.Thread(target=server.run, daemon=True)
    server_thread.start()
    for _ in range(30):
        if port_is_open(PORT):
            break
        time.sleep(1)
    if not port_is_open(PORT):
        raise RuntimeError("API server did not start. Check the errors above.")

ENGINE_URL = None

if "Ngrok" in TUNNEL_PROVIDER:
    from pyngrok import ngrok
    if not NGROK_AUTH_TOKEN.strip():
        from getpass import getpass
        NGROK_AUTH_TOKEN = getpass("Paste your Ngrok auth token: ").strip()
    if not NGROK_AUTH_TOKEN.strip():
        raise ValueError("Ngrok auth token is required for Ngrok provider.")
    try: ngrok.kill()
    except Exception: pass
    ngrok.set_auth_token(NGROK_AUTH_TOKEN.strip())
    tunnel = ngrok.connect(addr=PORT, proto="http")
    ENGINE_URL = tunnel.public_url.rstrip("/")
else:
    print("[*] Launching Cloudflare Quick Tunnel...")
    cf_proc = subprocess.Popen(
        ["cloudflared", "tunnel", "--url", f"http://127.0.0.1:{PORT}"],
        stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True
    )
    cf_pattern = re.compile(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com")
    t0 = time.time()
    while time.time() - t0 < 35:
        line = cf_proc.stderr.readline()
        if not line: continue
        match = cf_pattern.search(line)
        if match:
            ENGINE_URL = match.group(0).rstrip("/")
            break
    if not ENGINE_URL:
        raise RuntimeError("Cloudflare quick tunnel timed out. Check network or select Ngrok provider.")

# Verify public tunnel health
for attempt in range(15):
    try:
        resp = requests.get(ENGINE_URL + "/health", timeout=10)
        if resp.ok:
            break
    except requests.RequestException:
        pass
    time.sleep(2)
else:
    raise RuntimeError(f"Tunnel started at {ENGINE_URL}, but public /health check failed. Please re-run.")

print("\n" + "=" * 65)
print("🚀 AI SUBTITLE STUDIO COLAB ENGINE IS ONLINE!")
print("=" * 65)
print("  🔗 Engine URL : ", ENGINE_URL)
print("  🔑 API Token  : ", API_TOKEN)
print("  ⚡ Whisper     : ", WHISPER_MODEL)
print("  ⚡ Device      : ", DEVICE, f"({COMPUTE_TYPE})")
print("  📄 Health Check: ", ENGINE_URL + "/health")
print("=" * 65)
print("\n👉 Instructions for AI Subtitle Studio (http://localhost:3000):")
print("1. In the studio, open Settings (⚙️).")
print("2. Paste 'Engine URL' and 'API Token' into the Colab Addon section.")
print("3. Click 'Save & Verify Connection'.")
print("\n⚠️ Keep this cell running. Disconnecting Colab = tunnel goes offline.")

try:
    while True:
        time.sleep(1)
except KeyboardInterrupt:
    print("\n[!] Stopping server and tunnel...")

In [ ]:
# @title 4. Real speech & translation self-test (Optional)
# @markdown Creates spoken sample audio with espeak-ng, transcribes with Faster-Whisper, and tests OPUS-MT.
import subprocess, os, shutil

sample_text = "Hello. Welcome to AI Subtitle Studio. This is a real speech recognition test."
if not shutil.which("espeak-ng"):
    subprocess.check_call(["apt-get", "update", "-qq"])
    subprocess.check_call(["apt-get", "install", "-y", "-qq", "espeak-ng"])

print("[*] Generating synthetic speech WAV with espeak-ng...")
subprocess.run(["espeak-ng", "-s", "135", "-w", "colab_sample.wav", sample_text], check=True)
subprocess.run(["ffmpeg", "-y", "-i", "colab_sample.wav", "-ac", "1", "-ar", "16000", "-c:a", "pcm_s16le", "colab_sample_16k.wav"],
               check=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

print("[*] Transcribing audio with Faster-Whisper...")
model = get_whisper()
segments, info = model.transcribe("colab_sample_16k.wav", language="en", vad_filter=True)
recognized = " ".join(s.text.strip() for s in segments)
print("Reference :", sample_text)
print("Recognized:", recognized)
print("Language  :", info.language)
print("Device    :", DEVICE)
assert recognized.strip(), "Whisper returned no speech!"

print("\n[*] Testing OPUS-MT translation (Japanese -> English)...\n")
sample_ja = ["こんにちは、世界！", "AI字幕スタジオへようこそ。"]
translations = translate_opus(sample_ja, "ja", "en")
for orig, trans in zip(sample_ja, translations):
    print(f"  JA: {orig} -> EN: {trans}")

print("\n✅ ALL TESTS PASSED! Colab AI Engine is 100% verified and operational.")

## 🛠️ Troubleshooting Guide

- **GPU not detected:** In Colab, click **Runtime** → **Change runtime type** → select **T4 GPU** → click **Save**. If a free GPU is unavailable, the notebook falls back to CPU automatically.
- **CUDA out of memory (OOM):** Change `WHISPER_MODEL` to `small` or `base` in Cell 2, then re-run.
- **Upload too large (413 error):** AI Subtitle Studio automatically extracts 16 kHz mono audio and compresses files over 45 minutes to 64 kbps MP3 below 95 MB.
- **Translation pair unsupported:** The notebook translates configured language pairs (Japanese, French, German, Spanish, Chinese, Arabic, Hindi, Tamil, Telugu, Malayalam, Kannada) to English. It returns an explicit error for unsupported pairs instead of pretending the original text was translated.
- **Colab disconnects:** Re-run the cells; a new temporary URL and API token will be generated.